In [7]:
import pandas as pd
import requests
import json
from  tqdm.auto import tqdm

/Users/bajajn/gen-ai/llm-zoomcamp/00-search-engine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
courses_url = "https://datatalks.club/faq/json/courses.json"

In [4]:
response = requests.get(courses_url)
courses = response.json()

In [5]:
courses

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [8]:
documents = []
courses_faq_base_url = "https://datatalks.club/faq"
for course in tqdm(courses):
    course_faq_url = f"{courses_faq_base_url}{course['path']}"
    course_response = requests.get(course_faq_url)
    course_response.raise_for_status()
    course_faq_data = course_response.json()
    documents.extend(course_faq_data)

len(documents)

100%|██████████| 6/6 [00:01<00:00,  3.22it/s]


1342

In [9]:
data = pd.DataFrame(data=documents)
data.head()

,id,course,section,question,answer
0,9e508f2212,data-engineering-zoomcamp,General Course-Related Questions,Course: When does the course start?,A new cohort runs roughly January–April every ...
1,bfafa427b3,data-engineering-zoomcamp,General Course-Related Questions,Course: What are the prerequisites for this co...,"To get the most out of this course, you should..."
2,3f1424af17,data-engineering-zoomcamp,General Course-Related Questions,Course: Can I still join the course after the ...,"Yes, even if you don't register, you're still ..."
3,52217fc51b,data-engineering-zoomcamp,General Course-Related Questions,Course: I have registered for the Data Enginee...,You don't need a confirmation email. You're ac...
4,33fc260cd8,data-engineering-zoomcamp,General Course-Related Questions,Course: What can I do before the course starts?,Get the basic environment ready ahead of time:...


Vector spaces
- turn the docs into vectors
- term-document matrix:
  - rows: documents
  - columns: words/tokens
- bag of words
    - where does the word exist in the sentence is lost 

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

In [11]:
cv = CountVectorizer(min_df=3, stop_words='english')
cv.fit(data.answer)

,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"min_df min_df: float in range [0.0, 1.0] or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float, the parameter represents a proportion of documents, integerabsolute counts.This parameter is ignored if vocabulary is not None.",3
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"


In [13]:
processed_data = cv.transform(data.answer)

In [14]:
df_processed = pd.DataFrame(data=processed_data.todense(), columns=cv.get_feature_names_out())

In [15]:
df_processed.transpose().sample(50)

,0,1,2,3,4,5,6,7,8,9,...,1332,1333,1334,1335,1336,1337,1338,1339,1340,1341
asynchronous,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
detect,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
latest,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
line,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0
wrapper,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
users,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
codex,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
postgres_password,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
implement,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
check,1,0,0,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [17]:
tfidf = TfidfVectorizer(stop_words='english', min_df=3)

In [18]:
tfidf.fit(data.answer)

,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",3
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'


In [19]:
processed_data = tfidf.transform(data.answer)

In [20]:
df_processed = pd.DataFrame(data=processed_data.todense(), columns=cv.get_feature_names_out())

In [21]:
df_processed.head()

,00,000,001,01,02,02d,03,04,05,06,...,yourusername,youtube,yyyy,zero,zip,zone,zoomcamp,zoomcamp_test,zoomcamps,zsh
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.085531,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.119765,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


In [22]:
query = "Do i need to know python to sign up for a January course?"

processed_query = tfidf.transform([query])

In [23]:
processed_query.toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 2868))

In [24]:
q_dict = {i:j.item() for i,j in zip(tfidf.get_feature_names_out(), processed_query.toarray()[0])}

In [25]:
q_dict['python']

0.1987050613863148

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
scores = cosine_similarity(X=processed_data, Y=processed_query).flatten()

In [28]:
import numpy as np

In [29]:
np.argsort(scores)[-5:]

array([358,   0, 896, 907, 875])

In [31]:
for i in np.argsort(scores)[-5:]:
    print(data.answer[i]+"\n")

The Faust library is no longer maintained — see https://github.com/robinhood/faust. We don't recommend using it for the course.

If you don't know Java, follow the Python streaming materials in the course repo (under `06-streaming/python/`) instead, which use Kafka or Redpanda directly. Watching the Java videos to understand the streaming concepts is still useful even if you skip the coding parts.

A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.

Check [this article](https://mlbookcamp.com/article/python). If you know everything in this article, you know enough.

In [32]:
text_columns = [cols for cols in data if 'course' not in cols and 'id' not in cols]
matrices = {}
vectorisors = {}
for fields in text_columns:
    cv = TfidfVectorizer(stop_words='english', min_df=5)
    X = cv.fit_transform(data[fields])
    matrices[fields] = X
    vectorisors[fields] = cv

In [33]:
filters = {
    'course' : 'data-engineering-zoomcamp'
}

boost = {
    'question' : 3,
    'text' : 2
}

In [34]:
score = np.zeros(data.shape[0])
query = "Do i need to know python to sign up for a January course?"
for field in text_columns:
    processed_query = vectorisors[field].transform([query])
    f_score = cosine_similarity(matrices[field], processed_query).flatten()
    for f in filters:
        mask = (data[f] == filters[f]).astype(int)
    importance = boost.get(field, 1.0)
    score = (score + f_score*importance)*mask

In [35]:
score

0       1.312289
1       1.549006
2       0.999467
3       0.874248
4       1.527180
          ...   
1337    0.000000
1338    0.000000
1339    0.000000
1340    0.000000
1341    0.000000
Name: course, Length: 1342, dtype: float64

In [36]:
top_content = np.argsort(score)[-5:]

In [37]:
top_content

1337    215
1338     20
1339    105
1340      4
1341      1
Name: course, dtype: int64

In [38]:
for i in top_content:
    print(data.iloc[i].question)
    print()
    print(data.iloc[i].answer)
    print()
    print()

Python: Generators in Python

A generator is a function in Python that returns an iterator using the `yield` keyword.

A generator is a special type of iterable, similar to a list or a tuple, but with a crucial difference. Instead of creating and storing all the values in memory at once, a generator generates values on-the-fly as you iterate over it. This makes generators memory-efficient, particularly when dealing with large datasets.


How can we contribute to the course?

- [Star the repository](https://github.com/DataTalksClub/data-engineering-zoomcamp).
- Share it with friends if you find it useful.
- Create a pull request (PR) if you can improve the text or structure of the repository.
- [Update this FAQ](https://github.com/DataTalksClub/faq/).


Python: Python can't ingest data from the GitHub link provided using curl

```python
os.system(f"curl -LO {url} -o {csv_name}")
```


Course: What can I do before the course starts?

Get the basic environment ready ahead of time:

- Goog

**Building the Search Service class that represents the Elastic Search**

In [39]:
class SearchService:
    def __init__(self, text_fields: list[str]):
        self.data = pd.DataFrame()
        self.text_fields = text_fields
        self.matrices = {}
        self.vectorisors = {}

    def fit(self, records: dict, vectorisor_params: dict={}):
        self.data = pd.DataFrame(records)
        for fields in self.text_fields:
            cv = TfidfVectorizer(**vectorisor_params)
            X = cv.fit_transform(self.data[fields])
            self.matrices[fields] = X
            self.vectorisors[fields] = cv

    def search_top_results(self,
                           query: str,
                           filters: dict={},
                           boost: dict={},
                           num_results: int=5):

        score = np.zeros(self.data.shape[0])
        for fields in self.text_fields:
            processed_query = self.vectorisors[fields].transform([query])
            f_score = cosine_similarity(self.matrices[fields], processed_query).flatten()
            
            importance = boost.get(fields, 1.0)
            score = score + f_score*importance
        for f in filters:
            mask = (self.data[f] == filters[f]).astype(int)
            score = score*mask

        top_indexes = np.argsort(score)[-num_results:]
        results = self.data.iloc[top_indexes]
        return results.to_dict(orient='records') 

In [40]:
ss = SearchService(text_fields=text_columns)

In [41]:
ss.fit(
    data.to_dict(orient='records'),
    vectorisor_params={"stop_words": "english", "min_df": 5}
)

In [42]:
ss.search_top_results(
    query='I just signed up, is it too late to start the course?',
    num_results=5,
    boost={'question': 3},
    filters={'course': 'data-engineering-zoomcamp'}
)

[{'id': '721f9e0c29',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How can we contribute to the course?',
  'answer': '- [Star the repository](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n- Share it with friends if you find it useful.\n- Create a pull request (PR) if you can improve the text or structure of the repository.\n- [Update this FAQ](https://github.com/DataTalksClub/faq/).'},
 {'id': '33fc260cd8',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What can I do before the course starts?',
  'answer': "Get the basic environment ready ahead of time:\n\n- Google Cloud account (free trial — see the GCP setup FAQ).\n- Google Cloud SDK (`gcloud` CLI).\n- Python 3 — install via your OS package manager or [`uv`](https://docs.astral.sh/uv/) (recommended for managing Python versions and project venvs).\n- Terraform.\n- Git.\n\nThen look over the pre

**Now we will test with embeddings but just with text data**

In [45]:
from sklearn.decomposition import TruncatedSVD, NMF

In [43]:
cv = ss.vectorisors['answer']
X = ss.matrices['answer']

In [46]:
svd = TruncatedSVD(random_state=42, n_components=16)
X_emb = svd.fit_transform(X)

In [48]:
query = "I just signed up, is it too late to start the course?"
query_processed = cv.transform([query])
query_emb = svd.transform(query_processed)
score = cosine_similarity(X_emb, query_emb).flatten()
top_results = np.argsort(score)[-5:]
data.iloc[top_results].to_dict(orient='records')

[{'id': '9d57199de9',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Can I skip topics I already know?',
  'answer': 'Yes. All lesson content is optional; only the projects are mandatory. Move ahead at your own pace (you don’t need to wait for a “module start” date).'},
 {'id': '16005581f2',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Edit Course Profile.',
  'answer': 'The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to be a nickname or your real name if you prefer. Your entry on the Leaderboard is the one highlighted in light green.\n\nThe Certificate name should be your actual name that you want to appear on your certificate after completing the course.\n\nThe "Display on Leaderboard" option indicates whether you want your name to be listed on the course leaderboard.'},
 {'id': 'cd627dc856',
  'course': 'stock-markets-

In [49]:
class SearchServiceWithEmbeddings:
    def __init__(self, text_fields: list[str]):
        self.data = pd.DataFrame()
        self.text_fields = text_fields
        self.matrices = {}
        self.vectorisors = {}
        self.embeddors = {}

    def fit(self, records: dict, vectorisor_params: dict={}, max_embeddings: int=16):
        self.data = pd.DataFrame(records)
        for fields in self.text_fields:
            cv = TfidfVectorizer(**vectorisor_params)
            X = cv.fit_transform(self.data[fields])
            max_words = max([len(sentence.split(" ")) for sentence in self.data[field]])
            if max_words > 3*max_embeddings:
                svd = TruncatedSVD(random_state=42, n_components=max_embeddings)
                X = svd.fit_transform(X)
                self.embeddors[fields] = svd
            self.matrices[fields] = X
            self.vectorisors[fields] = cv

    def search_top_results(self,
                           query: str,
                           filters: dict={},
                           boost: dict={},
                           num_results: int=5):

        score = np.zeros(self.data.shape[0])
        for fields in self.text_fields:
            processed_query = self.vectorisors[fields].transform([query])
            emb_query = self.embeddors[fields].transform(processed_query) if self.embeddors.get(fields) else processed_query.copy()
            f_score = cosine_similarity(self.matrices[fields], emb_query).flatten()
            
            importance = boost.get(fields, 1.0)
            score = score + f_score*importance
        for f in filters:
            mask = (self.data[f] == filters[f]).astype(int)
            score = score*mask

        top_indexes = np.argsort(score)[-num_results:]
        results = self.data.iloc[top_indexes]
        return results.to_dict(orient='records') 

In [50]:
sse = SearchServiceWithEmbeddings(text_fields=text_columns)
sse.fit(
    data.to_dict(orient='records'),
    vectorisor_params={"stop_words": "english", "min_df": 5}
)

In [51]:
sse.search_top_results(query='I just signed up, is it too late to start the course?',
    num_results=5,
    boost={'question': 2},
    filters={'course': 'data-engineering-zoomcamp'}
)

[{'id': 'b71fb3b195',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: how many Zoomcamps run in a year?',
  'answer': 'DataTalks.Club runs several Zoomcamps every year. The roster and approximate timing:\n\n- Data Engineering: Jan – Apr\n- Stock Market Analytics: Apr – May\n- MLOps: May – Aug\n- LLM: Jun – Sep\n- Machine Learning: Sep – Jan\n\nFor the up-to-date list and current dates, see the [DataTalks.Club guide to free courses](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\nEach Zoomcamp has one "live" cohort per year — that\'s the only window in which you can earn the certificate. Outside the live cohort you can still take the course self-paced (materials stay open), but no certificate.'},
 {'id': '068529125b',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'answer': 

**Lets try with NMF**

In [52]:
class SearchServiceWithEmbeddings:
    def __init__(self, text_fields: list[str]):
        self.data = pd.DataFrame()
        self.text_fields = text_fields
        self.matrices = {}
        self.vectorisors = {}
        self.embeddors = {}

    def fit(self, records: dict, vectorisor_params: dict={}, max_embeddings: int=16):
        self.data = pd.DataFrame(records)
        for fields in self.text_fields:
            cv = TfidfVectorizer(**vectorisor_params)
            X = cv.fit_transform(self.data[fields])
            max_words = max([len(sentence.split(" ")) for sentence in self.data[field]])
            if max_words > 3*max_embeddings:
                svd = NMF(random_state=42, n_components=max_embeddings, max_iter=1000)
                X = svd.fit_transform(X)
                self.embeddors[fields] = svd
            self.matrices[fields] = X
            self.vectorisors[fields] = cv

    def search_top_results(self,
                           query: str,
                           filters: dict={},
                           boost: dict={},
                           num_results: int=5):

        score = np.zeros(self.data.shape[0])
        for fields in self.text_fields:
            processed_query = self.vectorisors[fields].transform([query])
            emb_query = self.embeddors[fields].transform(processed_query) if self.embeddors.get(fields) else processed_query.copy()
            f_score = cosine_similarity(self.matrices[fields], emb_query).flatten()
            
            importance = boost.get(fields, 1.0)
            score = score + f_score*importance
        for f in filters:
            mask = (self.data[f] == filters[f]).astype(int)
            score = score*mask

        top_indexes = np.argsort(score)[-num_results:]
        results = self.data.iloc[top_indexes]
        return results.to_dict(orient='records') 

In [53]:
sse = SearchServiceWithEmbeddings(text_fields=text_columns)
sse.fit(
    data.to_dict(orient='records'),
    vectorisor_params={"stop_words": "english", "min_df": 5}
)

In [46]:
sse.search_top_results(query='I just signed up, is it too late to start the course?',
    num_results=5,
    boost={'question': 2},
    filters={'course': 'data-engineering-zoomcamp'}
)

[{'text': "No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.",
  'section': 'General course-related questions',
  'question': 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "Yes, even if you don't register, you're stil

**Now we will try bert to get some semantics going**

In [54]:
import torch
from transformers import BertModel, BertTokenizer
from tqdm.auto import tqdm

ModuleNotFoundError: No module named 'torch'

In [48]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [49]:
my_list = [1,2,3,4,5,3,2,3,4,5,1]
def make_batches(seq, n):
    result = []
    for i in range(0, len(seq), n):
        batch = seq[i:i+n]
        result.append(batch)
    return result

# Testing my make batches

In [50]:
make_batches(my_list, 3)

[[1, 2, 3], [4, 5, 3], [2, 3, 4], [5, 1]]

In [51]:
def make_batches(seq, batch_size: int):
    result = []
    for i in range(0, len(seq), batch_size):
        batch = seq[i:i+batch_size]
        result.append(batch)
    return result

In [52]:
data = ["Andes wakes in the morning and calls his dad, saying `ba` `ba` `ga`",
        "Dad is always running late for the morning shift with the baby",
        "Andes the baby is ready to play as soon he wakes up",
        "Andes the baby doesnt enjoy the morning routine around changing nappy, brushing his teeth, and putting on clothes",
        "Mom wants Andes to eat something like a porridge, or some food, but Andes most days reject and just want mum's milk",
        "Andes loves to chase the cats and play with the ball"]

df = pd.DataFrame(data={"text":data})

In [53]:
df

,text
0,"Andes wakes in the morning and calls his dad, ..."
1,Dad is always running late for the morning shi...
2,Andes the baby is ready to play as soon he wak...
3,Andes the baby doesnt enjoy the morning routin...
4,Mom wants Andes to eat something like a porrid...
5,Andes loves to chase the cats and play with th...


In [54]:
def compute_embeddings(texts, batch_size=8):
    
    #Creating and running  in batches
    text_batches = make_batches(texts, 8)
    
    all_embeddings = []
    
    for batch in tqdm(text_batches):

        # Encoding the text to send to bert model
        encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
    
        with torch.no_grad():
            # Running the torch without gradient descent
            
            # Getting the output by calling the model
            outputs = model(**encoded_input)

            # Extracting the hidden state embeddings
            hidden_states = outputs.last_hidden_state

            # Getting the mean of the embeddings
            batch_embeddings = hidden_states.mean(dim=1)

            # Getting them in an array
            batch_embeddings_np = batch_embeddings.cpu().numpy()

            # Appending the embeddings
            all_embeddings.append(batch_embeddings_np)

    # Stacking all the batched embeddings, as a singular array
    final_embeddings = np.vstack(all_embeddings)
    return final_embeddings

In [55]:
X_text = compute_embeddings(df.text.to_list(), batch_size=2)

  0%|          | 0/1 [00:00<?, ?it/s]

In [56]:
query='What are morning tasks for dad?'
query_emb = compute_embeddings([query])

  0%|          | 0/1 [00:00<?, ?it/s]

In [57]:
score = cosine_similarity(X_text, query_emb).flatten()
top_results = np.argsort(-score)[:2]
df.iloc[top_results].to_dict(orient='records')

[{'text': 'Andes the baby doesnt enjoy the morning routine around changing nappy, brushing his teeth, and putting on clothes'},
 {'text': 'Dad is always running late for the morning shift with the baby'}]